# Kandinsky 6.0: video koşullu ses deneyi

Bu notebook **eğitim yapmaz**. Kandinsky 6.0’ın videoyu koşul olarak alan ses üretim yolunu tek bir oyun klibinde çalıştırır. Model video ile birlikte metin istemi de bekler; deneyimiz **video + aynı prompt** ile üretimdir.

## Adımlar
1. Colab’de **A100 GPU** seçip ön kontrolü çalıştır.
2. Kandinsky’nin resmî deposunu ve Lite Distill ağırlıklarını indir.
3. Üretmek istediğin süreyi kapsayan bir MP4 yükle.
4. Klipteki ses olayını bir cümleyle tarif edip üret.
5. Çıktıyı dinle, puanla ve MP4/JSON raporunu indir.

İlk turda tek seed kullanıyoruz. Buradaki hedef eğitim değil; Colab kurulumunu, video koşullu üretimi ve ürün açısından anlamlı bir örneği doğrulamak.

## Lisans ve kapsam

Kandinsky’nin resmî deposu ve Lite Distill model kartı MIT lisansı bildiriyor. Ticari entegrasyondan önce indirilen checkpoint revizyonunu ve bütün yardımcı ağırlıkların/bağımlılıkların lisansını ayrıca envanterleyeceğiz. Bu ilk deney ticari onay değildir.

Ayrıca model videoyu görür ama **metin istemi de gerektirir**. Dolayısıyla bu, saf video-girdili bir deney değil; videonun prompta ek katkısını sınar.

Kaynaklar: [resmî repo](https://github.com/kandinskylab/kandinsky-6), [Lite Distill ağırlıkları ve model kartı](https://huggingface.co/kandinskylab/Kandinsky-6.0-Lite-distill-5s-Diffusers), [V2A kullanım kılavuzu](https://github.com/kandinskylab/kandinsky-6/blob/main/docs/generation_modes.md).

In [ ]:
# Ön kontrol: Kandinsky resmî kurulum profili T4 GPU içermiyor; A100 önerilir.
import os, sys, subprocess
print('Python:', sys.version)
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,compute_cap', '--format=csv,noheader'], capture_output=True, text=True)
print(gpu.stdout or gpu.stderr)
if gpu.returncode != 0:
    raise RuntimeError('Colab Runtime > Change runtime type > GPU seç ve yeniden çalıştır.')
if 'A100' not in gpu.stdout:
    print('Bu notebook A100 için hazırlandı. Resmî Kandinsky GPU kurulum betiği T4 (7.5) için profil tanımlamıyor.')
if sys.version_info < (3, 13):
    raise RuntimeError('Resmî Kandinsky pipeline Python 3.13 veya 3.14 istiyor. Bu runtime sürümüyle kuruluma devam etmeyin.')

## 1 — Resmî çalışma ortamını kur

Kandinsky’nin kendi GPU kurulum betiği A100 için SageAttention kurmayı dener; Colab ortamında bu derleme her zaman kullanılabilir olmayabilir. İlk pilotta aşağıdaki config, ek paket gerektirmeyen PyTorch SDPA dikkat mekanizmasını kullanır. API anahtarı gerekmiyor.

In [ ]:
# Resmî depoyu indir ve GPU'ya uygun ortamı kur.
%cd /content
!git clone --depth 1 https://github.com/kandinskylab/kandinsky-6.git
%cd /content/kandinsky-6
!python -m pip -q install uv
!curl -fsSL https://just.systems/install.sh | bash -s -- --to /usr/local/bin
!just setup
print('Kaynak commit:', subprocess.check_output(['git','rev-parse','HEAD'], text=True).strip())

## 2 — Lite Distill checkpoint’ini indir

İlk deneme için 3B parametreli ve 10 adımlı Lite Distill sürümünü kullanıyoruz. Bu notebook model eğitmez veya fine-tune etmez. Ağırlıklar büyük olduğundan Colab diskinde yeterli boş alan bırak.

In [ ]:
!just download lite-distill
print('Lite Distill checkpoint indirildi.')

## 3 — Test klibini yükle

En az 5 saniyelik MP4 seç. Örnekleme hücresi dosyanın tamamını saklar, yalnızca önizleme için ilk 5 saniyeyi gösterir. Üretim hücresindeki süre ayarı modele hangi aralığın verileceğini belirler. Kaynak ses modele verilmez; yalnızca video kareleri koşul olarak kullanılır.

In [ ]:
from google.colab import files
from pathlib import Path
from IPython.display import display, Video, Audio
import json, time, subprocess, hashlib

uploaded = files.upload()
if not uploaded:
    raise RuntimeError('Bir MP4 video yüklemedin.')
filename = next(iter(uploaded))
# files.upload() saves into the current working directory, which may be /content/kandinsky-6.
# Write the returned bytes explicitly so the input path is stable regardless of the current directory.
safe_filename = Path(filename).name
source_video = Path('/content') / safe_filename
source_video.write_bytes(uploaded[filename])
probe = subprocess.run(['ffprobe','-v','error','-show_entries','format=duration','-of','default=noprint_wrappers=1:nokey=1',str(source_video)], capture_output=True, text=True)
if probe.returncode != 0:
    raise RuntimeError('Video okunamadı: ' + probe.stderr[-1000:])
source_duration = float(probe.stdout.strip())
print(f'Kaynak klip: {filename} | {source_duration:.2f} saniye')
if source_duration < 5.0:
    raise ValueError('Bu checkpoint için en az 5 saniyelik video yükle.')

preview_video = Path('/content/kandinsky_v2a_preview.mp4')
subprocess.run(['ffmpeg','-y','-i',str(source_video),'-t','5','-map','0:v:0','-an','-c:v','libx264','-pix_fmt','yuv420p','-r','24',str(preview_video)], check=True, capture_output=True)
print('Önizleme (ilk 5 saniye):')
display(Video(str(preview_video), embed=True))

## 4 — Video + metin istemiyle ses üret

İstemi yüklediğin klibe göre düzenle ve yalnızca görülen ses olayını tarif et. Aynı cümleyi Stable Audio kolunda da kullanabiliriz. Böylece **Stable: prompt** ile **Kandinsky: video + aynı prompt** karşılaştırması yapılabilir. Sabit seed, bu ilk denemeyi yeniden üretilebilir kılar.

A100 için depodaki hazır preset Pro Distill ve SageAttention seçiyor. Bu deney Lite Distill indirdiği için aşağıdaki kod Lite checkpoint'ini seçen bir A100 config'i oluşturur. SageAttention Colab'de yoksa yerleşik PyTorch SDPA kullanılır.

Checkpoint 5 saniyelik olsa da resmî CLI --time-length parametresini destekliyor. Varsayılanı 5 bırak; önce aynı klipte 10 saniyeyi ayrı bir deney olarak dene. 5 saniyenin üzerindeki kalite ve VRAM ihtiyacı garanti edilmiş değil.

In [ ]:
# Bunu yüklediğin klibe göre değiştir.
EVENT_PROMPT = 'A single wooden block falls and hits a hard wooden floor, producing one short hollow thud. No music, no speech.'
SEED = 42
GENERATION_SECONDS = 5  # İlk doğrulama: 5. Daha uzun tek çağrı denemesi için 10 yap.
if source_duration < GENERATION_SECONDS:
    raise ValueError(f'Kaynak video {GENERATION_SECONDS} saniyeden kısa: {source_duration:.2f}s')
input_video = Path('/content/kandinsky_v2a_input.mp4')
prep = subprocess.run(['ffmpeg','-y','-i',str(source_video),'-t',str(GENERATION_SECONDS),'-map','0:v:0','-an','-c:v','libx264','-pix_fmt','yuv420p','-r','24',str(input_video)], capture_output=True, text=True)
if prep.returncode != 0:
    raise RuntimeError('Üretim videosu hazırlanamadı: ' + prep.stderr[-1500:])
# A100 presetini Lite Distill ile eşleştir. Depodaki a100-80gb.yaml, Pro Distill seçer.
DEVICE_CONFIG = Path('/content/kandinsky-6/kandinsky/configs/devices/a100-lite-distill.yaml')
DEVICE_CONFIG.write_text('checkpoint: lite-distill\ncache:\n  mode: none\noffload:\n  strategy: module\ncompile:\n  strategy: none\nattention:\n  engine: sdpa\ntext_embedder:\n  quantized_qwen: false\nbeautifier:\n  name: qwen25\n', encoding='utf-8')
OUTPUT_MP4 = Path('/content/kandinsky_v2a_output.mp4')

# CLI override makes the choice explicit even if a stale YAML config is present.
cmd = ['just','generate',EVENT_PROMPT,'--config',str(DEVICE_CONFIG),'--attention-engine','sdpa','--video-file',str(input_video),'--time-length',str(GENERATION_SECONDS),'--out',str(OUTPUT_MP4),'--seed',str(SEED)]
print('Üretim başlıyor. Komut:', ' '.join(cmd))
started = time.time()
result = subprocess.run(cmd, cwd='/content/kandinsky-6', text=True, capture_output=True)
print(result.stdout[-6000:])
if result.returncode != 0:
    print(result.stderr[-6000:])
    raise RuntimeError('Üretim başarısız oldu; hata metnini raporla.')
if not OUTPUT_MP4.exists():
    raise FileNotFoundError(f'Beklenen çıktı bulunamadı: {OUTPUT_MP4}')
elapsed_seconds = time.time() - started
print(f'Üretim süresi: {elapsed_seconds:.1f} saniye')

## 5 — Dinle ve puanla

Önce videoyu, sonra sesi dinle. **Olay eşleşmesi**, **zamanlama**, **ses kalitesi** ve **istenmeyen müzik/konuşma** için puan ver. Tek seed, modelin genel performansını kanıtlamaz; bu ilk teknik pilot.

In [ ]:
# Bu hücre yalnızca üretim tamamlandıktan sonra çalıştırılır.
from pathlib import Path
from IPython.display import display, Video, Audio
import subprocess

OUTPUT_MP4 = Path('/content/kandinsky_v2a_output.mp4')
if not OUTPUT_MP4.is_file():
    raise FileNotFoundError(f'Üretilen video bulunamadı: {OUTPUT_MP4}. Önce 4. üretim hücresini çalıştır.')

display(Video(data=OUTPUT_MP4.read_bytes(), embed=True, mimetype='video/mp4'))
OUTPUT_WAV = Path('/content/kandinsky_v2a_audio.wav')
extract = subprocess.run(['ffmpeg','-y','-i',str(OUTPUT_MP4),'-vn','-ac','2','-ar','44100',str(OUTPUT_WAV)], capture_output=True, text=True)
if extract.returncode != 0:
    raise RuntimeError('Videodan ses çıkarılamadı: ' + extract.stderr[-2000:])
if not OUTPUT_WAV.is_file() or OUTPUT_WAV.stat().st_size == 0:
    raise RuntimeError('Ses WAV dosyası oluşturulamadı veya boş.')
display(Audio(filename=str(OUTPUT_WAV)))

## 6 — Değerlendirme raporunu indir

Dinleme notlarını ve puanlarını doldur. Son hücre rapor JSON'unu üretir ve video ile birlikte tek ZIP dosyasında indirir; böylece tarayıcının birden çok indirmeyi engellemesi önlenir.

In [ ]:
# Dinleme sonrası bu alanları doldur.
from google.colab import files
from pathlib import Path
import json, hashlib, subprocess, zipfile

scores = {
    'visual_event_match_1_to_5': None,
    'audio_video_sync_1_to_5': None,
    'audio_quality_1_to_5': None,
    'unwanted_music_or_speech': None,
    'listening_notes': ''
}
report = {
    'experiment': 'kandinsky_v2a_colab_pilot_v1',
    'model': 'kandinskylab/Kandinsky-6.0-Lite-distill-5s-Diffusers',
    'source_repo': 'https://github.com/kandinskylab/kandinsky-6',
    'source_commit': subprocess.check_output(['git','-C','/content/kandinsky-6','rev-parse','HEAD'], text=True).strip(),
    'input_video_name': filename,
    'input_video_sha256': hashlib.sha256(source_video.read_bytes()).hexdigest(),
    'event_prompt': EVENT_PROMPT,
    'seed': SEED,
    'duration_seconds': GENERATION_SECONDS,
    'gpu': gpu.stdout.strip(),
    'generation_seconds': round(elapsed_seconds, 2),
    'scores': scores,
    'commercial_clearance': 'pending_full_dependency_and_data_provenance_review'
}
REPORT = Path('/content/kandinsky_v2a_report.json')
REPORT.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')
print(json.dumps(report, ensure_ascii=False, indent=2))
BUNDLE = Path('/content/kandinsky_v2a_bundle.zip')
with zipfile.ZipFile(BUNDLE, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    archive.write(OUTPUT_MP4, arcname=OUTPUT_MP4.name)
    archive.write(REPORT, arcname=REPORT.name)
files.download(str(BUNDLE))